# От классической корреляции к PyTorch Conv2d



**Цель:** Сопоставить cv2.filter2D и torch.nn.Conv2d, вычислить параметры и визуализировать карты признаков.

**Входные данные:** `images/source/coins.png`, `images/source/astronaut.png`

**Формула:** P = C_out·(C_in·K_h·K_w + 1) при bias=True.

## Последовательность выполнения

1. Преобразовать coins.png в float32/[0,1] и тензор (N,C,H,W)=(1,1,303,384).
2. Создать nn.Conv2d(1,1,3,padding=1,bias=False) и явно присвоить весам матрицу Sobel X.
3. Получить отклик CNN и cv2.filter2D с BORDER_CONSTANT; вычислить max |Δ| и объяснить сходство результатов.
4. Создать банк 4 фильтров (Sobel X, Sobel Y, тождественный, Laplace), показать четыре карты признаков.
5. Рассчитать число параметров для RGB→32, kernel=3×3, bias=True. Вывести shape(output) при входе (1,3,128,128), stride=1 и padding=1.
6. Сравнить масштаб параметров локального Conv2d и Dense слоя, который напрямую связывает каждый пиксель 128×128×3 со всеми выходными 32×128×128.

**Ожидаемая проверка:** Эталон: max difference между PyTorch и OpenCV в float32 близка к нулю; для Conv2d(3,32,3,bias=True) число параметров 896.

**Оценивание / 10:** Тензоры — 2; сверка двух реализаций — 3; банк фильтров — 2; параметры/вывод — 3.

**Вопросы:**
- Почему PyTorch называет Conv2d операцию, вычисляющую кросс-корреляцию?
- В чём выигрыш разделяемых весов?

In [ ]:
from pathlib import Path
import numpy as np, cv2, torch
R = Path.cwd() if (Path.cwd()/'images').exists() else Path.cwd().parent
im=cv2.imread(str(R/'images/source/coins.png'),0).astype(np.float32)/255.0
x=torch.from_numpy(im)[None,None]
K=np.array([[-1,0,1],[-2,0,2],[-1,0,1]],dtype=np.float32)
conv=torch.nn.Conv2d(1,1,3,padding=1,bias=False)
with torch.no_grad():conv.weight.copy_(torch.from_numpy(K)[None,None])
y=conv(x).detach().numpy()[0,0]
# TODO: compare with filter2D, 4 filters, calculate model params and output sizes.
print('Input:',tuple(x.shape),'Output:',y.shape)


## Ваш результат

1. Вставьте итоговые графики/выводы и таблицы в отчёт.
2. Укажите версии пакетов, время выполнения, выбранные параметры.
3. Сделайте краткий вывод: что работало, где ошибка, почему?